# Bonus 06 — The Lab 6 Pipeline, as a Library (LlamaIndex)

**Optional | After Lab 6 | Colab CPU | `OPENAI_API_KEY`**

---

In Lab 6 you built RAG by hand: split the documents, embed the chunks with MiniLM, store them in Chroma, retrieve, and write the grounding prompt yourself. Frameworks like **LlamaIndex** do all of that in a few lines.

The point of this notebook is not that the library is better. It is to feel what the abstraction hides, so you can choose it on purpose. We use the **same** embedding model as Lab 6 (local MiniLM, free) and the same `gpt-4o-mini`, so the only thing that changes is how much code you write, and how much you can still see.

```
Lab 6, by hand                          This notebook
RecursiveCharacterTextSplitter    →     Settings.chunk_size
MiniLM + Chroma                   →     Settings.embed_model + VectorStoreIndex
your RAG_PROMPT + rag()           →     index.as_query_engine()
```

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} llama-index llama-index-embeddings-huggingface openai python-dotenv

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: read the Secret you added
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv             # local: read .env in the repo root
    load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Add OPENAI_API_KEY as a Colab Secret or to .env"
print("key found; LlamaIndex reads it from the environment")

## 1. Tell LlamaIndex which models to use

LlamaIndex's defaults are OpenAI for both the chat model *and* the embeddings, which would bill you for every chunk you index. `Settings` overrides both, once, for everything that follows. Here: MiniLM for embeddings, exactly as in Lab 6, and `gpt-4o-mini` for answers.

`chunk_size` is in tokens, not characters. 256 tokens is roughly the 400-to-1,000-character range you chose between in Lab 6.

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai import OpenAI

Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.llm         = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.chunk_size  = 256
print("embeddings: MiniLM (local, free) | answers: gpt-4o-mini")

## 2. A small corpus

Four short files, written by the next cell if they are missing. (On Colab the repo's `Bonus/sample_docs/` folder is not there, so the notebook makes its own.)

In [ ]:
from pathlib import Path

SAMPLE = Path("sample_docs")
SAMPLE.mkdir(exist_ok=True)
docs_inline = {
    "quantization.md": "Quantization reduces weight precision. NF4 is 4-bit for LLM weights and uses less VRAM than FP16.",
    "rag.md": "RAG retrieves chunks at inference time and asks the model to answer only from that context. Use it for changing knowledge.",
    "lora.md": "LoRA trains small adapter matrices on a frozen base. QLoRA adds a 4-bit base so fine-tuning fits a smaller GPU.",
    "serving.md": "OpenAI-compatible APIs swap backends via base_url. vLLM adds PagedAttention and continuous batching for throughput.",
}
for name, text in docs_inline.items():
    p = SAMPLE / name
    if not p.exists():
        p.write_text(text)
print("files:", sorted(p.name for p in SAMPLE.iterdir()))


## 3. Load, index, ask

Three calls. `SimpleDirectoryReader` picks a parser per file type (text, markdown, PDF, Word). `VectorStoreIndex.from_documents` chunks, embeds and stores. `as_query_engine` wraps retrieval and generation into one `query()`.

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader(str(SAMPLE)).load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=2)

response = query_engine.query("When should I use RAG instead of fine-tuning?")
print(response)

**Checkpoint:** a grounded answer in three lines of code. Lab 6 took a whole notebook to get here. Now look at what you gave up seeing.

## 4. Open the box

**What was retrieved?** Every response keeps its sources. Lab 6 printed Chroma distances; here you get similarity scores and file names. Always look: this is where you debug a bad answer.

In [ ]:
for node in response.source_nodes:
    print(f"{node.score:.3f}  {node.metadata.get('file_name')}: {node.text[:90]}")

**What prompt did the model see?** In Lab 6 you wrote `RAG_PROMPT` yourself. LlamaIndex wrote one for you. Here it is:

In [ ]:
print(query_engine.get_prompts()["response_synthesizer:text_qa_template"].get_template())

**Checkpoint:** "Given the context information and not prior knowledge, answer the query." That is nearly Lab 6's grounding instruction, with one difference: it says nothing about what to do when the context does *not* cover the question. Lab 6 found that wording changes how often the model refuses or improvises. With a library, you inherit its choice unless you know where to look. (`query_engine.update_prompts(...)` replaces it.)

## 5. Keep the index

In-memory indexes die with the runtime. Persisting writes the chunks and vectors to a folder, like Lab 6's `chroma_db`, so you can load them later without re-embedding.

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

index.storage_context.persist(persist_dir="./vector_db_llama")
reloaded = load_index_from_storage(StorageContext.from_defaults(persist_dir="./vector_db_llama"))
print(reloaded.as_query_engine().query("What is QLoRA?"))

**Checkpoint:** the QLoRA answer mentions adapters and a 4-bit base, loaded from disk.

## 6. Your own documents

Upload one or two PDFs into a folder called `student_pdfs` (Colab: Files panel → right-click → New folder, then upload), and point the reader at it.

In [ ]:
from pathlib import Path

if Path("student_pdfs").exists():
    my_index = VectorStoreIndex.from_documents(SimpleDirectoryReader("student_pdfs").load_data())
    print(my_index.as_query_engine().query("What is the main topic of these documents?"))   # TODO: ask something specific
else:
    print("No student_pdfs folder yet. Create it and upload a PDF, then rerun this cell.")

---

## Library or by hand?

| | Lab 6 (by hand) | LlamaIndex |
|---|---|---|
| Lines of code to a first answer | a notebook | about five |
| Chunking | you choose splitter and size | `Settings.chunk_size`, or a custom node parser |
| Hybrid search (Lab 6 Part D) | twenty lines you understand | available, but you configure its retrievers |
| The grounding prompt | you wrote it | written for you; read it with `get_prompts()` |
| When something is wrong | you know every step | you need to know where the library hides each step |

Use the library when its defaults fit and you know how to open it up. Drop to Lab 6's approach when you need control over a stage, or when a bad answer means you have to see every step.

## Bonus 06 complete

- [ ] An index built with the same MiniLM as Lab 6, at no embedding cost
- [ ] Sources printed for an answer
- [ ] The hidden prompt read, and compared with Lab 6's
- [ ] An index saved to disk and loaded back

Next: [Bonus 07 — Hugging Face Spaces](07_hf_spaces_deployment.md) keeps the Lab 7 app online after class.